# Gene-level Profiling Analysis

This is the 2nd step of the actual data analysis, and is used to visualize similarity between samples and conditions. <br>
**Run preprocessing & RiboSeq_MetageneAnalysis.ipynb first!**

Requires **raw** WIG files (for all Ribo & RNA seq samples), genome fasta/gb, & mapping offset chosen during Metagene analysis. **User must specify condition information** and can optionally specify RiboSeq-RNAseq pairs for translation efficiency normalization

Also outputs .csv datatable with RPKM (&TE) normalization + log2FC & p-values by gene for each sample/condition

Runtime ~5min

Can be run simultaneously with RiboSeq_CodonAnalysis.ipynb


## Imports

In [ ]:
import os
import time
import numpy as np
import copy

import pandas as pd

from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.stats

import re

from importlib import reload

import RiboSeq_Analysis_fxns as analysis
import RiboSeq_Analysis_plots as plot


starttime = time.time()

## User inputs

In [ ]:
user_inputs = { 
    
    # list names of all files to be analyzed - must have existing WIG files 
    # includes both RiboSeq & RNAseq
    'filenames': [ 'Rb1_DMSO', 'Rb1_TZD-10',  'Rb2_DMSO', 'Rb2_TZD-10', 
                   'Rn1_DMSO', 'Rn1_10xTZD',  'Rn2_DMSO', 'Rn2_10xTZD',  ],

    # currently only used to get a count of how many genes pass filters for all samples in a given replicate
    # but all metagene/gene plots after filter step can be easily modified to give replicate-specific figures using the genes_per_repetition dict
    'repNums': {'Rb1': ['Rb1_DMSO', 'Rb1_TZD-10', ], 
                'Rb2': ['Rb2_DMSO', 'Rb2_TZD-10', ],
                'Rn1': ['Rn1_DMSO', 'Rn1_10xTZD', ],
                'Rn2': ['Rn2_DMSO', 'Rn2_10xTZD', ], }, #{},

    'rnaSeq_pairs': {'Rb1_DMSO': 'Rn1_DMSO', 'Rb1_TZD-10': 'Rn1_10xTZD', 
                     
                      'Rb2_DMSO': 'Rn2_DMSO', 'Rb2_TZD-10': 'Rn2_10xTZD', 
                      
                    
                    'DMSO': 'Rn_DMSO', 'TZD-10': 'Rn_TZD-10', } , # { ribo1: rna1 },

    # must have a condition labeled either WT or DMSO to use as comparison for log fold change
    'condition_info': { 'DMSO': ['Rb1_DMSO', 'Rb2_DMSO'], 'TZD-10': ['Rb1_TZD-10','Rb2_TZD-10'], 
                        
                       'Rn_DMSO': ['Rn1_DMSO', 'Rn2_DMSO'], 'Rn_TZD-10': ['Rn1_10xTZD','Rn2_10xTZD'],
                        } , 
    
    # for each condition in condition info, label with either 'RiboSeq' or 'RNAseq'
    'condition_type': { 'DMSO': 'RiboSeq', 'TZD-10': 'RiboSeq', 
                        
                       'Rn_DMSO': 'RNAseq', 'Rn_TZD-10': 'RNAseq',
                        },
    
    # main directory for analysis & containing preprocessing files; analysis folder will be generated here
    'main_dir': '/wynton/home/fujimori/jkleinman/JKDF05_Rb-Rn/', 
    
    # directory containing your WIG files from preprocessing analysis for all samples in filenames
    'wig_dir':  '/wynton/home/fujimori/jkleinman/JKDF05_Rb-Rn/preprocessing/Wig/',

    # path of the genome fasta file - must be same fasta file as was used for genome alignment in preprocessing step
    'genome_fasta': '/wynton/home/fujimori/jkleinman/RiboSeq_code/genome_files/CP009273.1/CP009273.1.fasta', 

    # path of the genbank file which matches the genome fasta file; features will be extracted from this
    'genome_genbank': '/wynton/home/fujimori/jkleinman/RiboSeq_code/genome_files/CP009273.1/CP009273.1.gb', 
    
    # arbitrary value; used during visualization but not included in total sequence depth or avg gene reads so won't be included in normalization factor
    'utr_length_to_include': 50,
    
    # choose if you're using the Wig pileup at 5'/3' end of read or at center of read
    # In my experience 3pr gives much better peaks for both start & stop codon - also favored for bacterial riboSeq (Mohammad, Green, & Buskirk 2019)
    'alignment_type': '3pr', #'5pr' 'avg' or '3pr'
    
    # used for assigning P site; mapping offset =0 uses raw wig data
    # adjust the value after running the analysis so start codon peak aligns with 0 and stop codon peak aligns with -6 (stop codon in A-site at -3)
    # Note: mapping offset is generally a negative value -- JKDF05 riboSeq places P-site with -15 offset for 3pr pileup on both metagene plots
    'mapping_offset': -15,
    
    # used to plot normalized reads across a given window for a gene of interest -- ('geneName', [min, max])
    # note that if you include UTRs they must be ≤ the value for utr_length_to_include 
    # providing an empty list will give the full CDS window for the gene (no UTR) -- ('geneName', [])
    # 'genes_of_interest': [] 
    'genes_of_interest': [ ] ,

    # how long your shortest CDS can be
    'minCDS': 18,

    # number of nt at either end of genes to exclude from codon analysis
    'exclude_end_nt': 9
    
            }

In [ ]:
if len(user_inputs['rnaSeq_pairs'].keys()) != 0:
    RNAseq = True
else:
    RNAseq = False

## Run the analysis

#### 1. Build the output directories

In [ ]:
# check for & build the analysis directories

analysis_outdir = user_inputs['main_dir'] + time.strftime("%Y%m%d") + '_analysis/'
offset_dir = analysis_outdir + '{}_offset_{}/'.format(user_inputs['alignment_type'], user_inputs['mapping_offset'])
profiling_dir = offset_dir  + 'geneProfiling/'
plot_dir = profiling_dir + 'plots/'
gene_dir = profiling_dir + 'geneWindow_plots/'


if not os.path.exists(analysis_outdir):
    print('Recommend running RiboSeq_MetageneAnalysis.ipynb first to choose appropriate mapping offset')
    os.mkdir(analysis_outdir)
if not os.path.exists(offset_dir):
    os.mkdir(offset_dir)
if not os.path.exists(profiling_dir):
    os.mkdir(profiling_dir)
if not os.path.exists(plot_dir):
    os.mkdir(plot_dir)
if not os.path.exists(gene_dir):
    os.mkdir(gene_dir)


#### 2. load genome and Wig files

> Same as for Metagene analysis
> - CDS features only & excluding pseudogenes
> - UTRs of length utr_length_to_include on either end, not going to be used for this analysis

>  - genome_dict 
>    - sequence dict contains full gene sequence with UTRs:
>      - full_sequence_dict = { featureName1 : [...ATG...TAG...], featureName2 : [...ATG...TAG...], ... } 
>    - strand dict contains info on + vs - strand for each gene:
>      - strand_dict = { featureName : 1, featureName2 : -1, ... }
>    - location dict contains genome positions of gene beginning & end:
>      -  location_dict = { featureName1 : (start, end),  ... }
>      -  not including UTRs
>    - total length of genome
>      -  total_genome_len = 4631469 (ex for CP009273.1)
> - feature_dict_meta = { sample_name1:  {gene1: [reads by position], gene2: [], ...}, sample_name2: {gene1: [reads by position], gene2: [], ... }
>   - includes readcounts for specified UTR length at both 5' and 3' ends

In [ ]:
### Load genome 

genome_dict = analysis.parse_genome(user_inputs)

# sanity check that gene info dictionaries are the same size 
print('CDS feature count (excluding pseudogenes:', len(genome_dict['full_sequence_dict']))
print('gene info dictionaries are same size:', len(genome_dict['full_sequence_dict'].keys()) == len(genome_dict['strand_dict'].keys()) == len(genome_dict['location_dict'].keys()))

### Load WIGs 

sample_files = {}

for fname in user_inputs['filenames']:
    sample_files[fname] = (user_inputs['wig_dir'] + fname +'_%s_fwd_fromSam.wig' % user_inputs['alignment_type'], 
                           user_inputs['wig_dir'] + fname +'_%s_rev_fromSam.wig' % user_inputs['alignment_type'])

sample_names = sample_files.keys()

### feature_dict_meta = { sample_name: {gene1:[reads by position], gene2: [], ...}
feature_dict_meta = analysis.read_in_wig_addOffset(sample_files, genome_dict, user_inputs)


#### 3. Calculate sample sequencing depth for normalization

> Calculate sequencing depth (total readCounts across entire genome) for each sample
>   - Using **CDS portions only** (exclude UTRs for purpose of normalization to read depth)
>   - value should be somewhat smaller than total readcount from bowtie alignment due to this restriction
>
> *Any reads in overlapping gene regions will be double counted -- hopefully relatively minimal effect*

In [ ]:
# check total read numbers across CDS portions only -- this will be the value used for normalization
# should be somewhat smaller than total read counts from WIG conversion in your preprocessing logs

utr_length_to_include = user_inputs['utr_length_to_include']

total_read_dict = {}
for sample in sample_names:
    all_features = []
    for read_list in feature_dict_meta[sample].values():
        all_features.extend(read_list[utr_length_to_include:-1*utr_length_to_include]) #read counts only in coding region - exclude UTR
    print(sample, np.sum(all_features))
    total_read_dict[sample] = np.sum(all_features)


#### 4. Create a table of RPKM counts

> **RPKM = reads per kilobase per million : normalizes read count to gene length and total sequencing depth**

> $$RPKM = \frac{numGeneReads}{GeneLength/1000*numTotalReads/1,000,000} = \frac{numGeneReads *10^9}{GeneLength*numTotalReads} $$
> `numGeneReads` = sum of reads mapped to a given gene <br>
> `GeneLength` = lenght of gene sequence (in bases)<br>
> `numTotalReads` = total number of mapped reads for the sample
>
> Also replace any/all "infinite" values with NaN
>
> **Note: currently working by GENE not by codon/nt**
>
> ** in all analyses below, inf & -inf values will be replaces with NaN

In [ ]:
# Calculate RPKM for ALL samples
df_master = analysis.geneRPKM(sample_names, feature_dict_meta, total_read_dict, genome_dict, user_inputs)
# Show the df to check
df_master

#### 5. Calculate Translation Efficiency

>If user has supplied RiboSeq-RNAseq pairs, normalize to total RNA -- Translation Efficiency (TE)
> - if rnaSeq_pairs = [ ] , downstream calculations will use RPKM instead
>
>Defining Translation Efficiency (TE) here as: `RPKM_ribo/RPKM_rna`


In [ ]:
# if user has supplied RiboSeq-RNAseq pairs, calculate translation efficiency TE
if RNAseq == True:
    df_master = analysis.TE(df_master, total_read_dict, user_inputs)
    
df_master

#### 6. Calculate condition averages & fold change compared to WT

 $$log2 (\frac{avgCondition}{avgWT}) $$

In [ ]:
df_master = analysis.conditionAvg(df_master, user_inputs, RNAseq)
df_master = analysis.logFoldChange(df_master, user_inputs, RNAseq)
df_master

#### 7. Assessing the statistical significance of differences in TE (or RPKM) using independent t-tests for each gene

>For each gene, compare between all samples in condition, versus all samples in WT:
>
> e.g. [Rb1_DMSO_RPKM, Rb2_DMSO_RPKM] vs [Rb1_TZD-100_RPKM, Rb2_TZD-100_RPKM]
>
> Raw p-value from this calculation will subsequently be adjusted for multiple testing
>
> ** assuming equal variance in the T-test which may/may not be correct


In [ ]:
df_master = analysis.independent_t_test(df_master, user_inputs, RNAseq)
df_master


#### 8. Multiple testing
> converting each p-value into a FDR corrected p-value using "Benjamini-Hochberg" procedure <br>
> https://islp.readthedocs.io/en/latest/labs/Ch13-multiple-lab.html <br>
> from Mangano code

In [ ]:
# note: this block throws an error due to the way p-values are assigned to the df
# will continue to run, so do not halt
df_master= analysis.multipleTest(df_master, user_inputs, RNAseq)
df_master

#### 9. Save df to .csv file for offline analysis
> code does not use this file, but useful to have

In [ ]:
# save the dataframe to csv
df_master.to_csv('{}/gene_profiling_master_table_{}_offset_{}.csv'.format(profiling_dir, user_inputs['alignment_type'], user_inputs['mapping_offset']))

#### 10. Sort by significance in each condition
> sort on raw p-value & display 10 most significant genes for each condition
>
> save the top 5 genes for whole gene-window plotting

In [ ]:
# sorting by significant genes for each condition (via raw pvalues) & just showing the top 10 

if RNAseq == False:
    datatype = 'RPKM'
else:
    datatype = 'TE'

WT = ['DMSO' if 'DMSO' in user_inputs['condition_info'] else 'WT'][0]
expConditions = [condition  for condition in user_inputs['condition_info'] if (condition != WT and user_inputs['condition_type'][condition] =='RiboSeq')]
significant_genes = []
    
for condition in expConditions:
    
    print('sorted by condition:', condition)
    df_master.sort_values(condition + '_%s_rawTtest_pval' % datatype, inplace=True)
    cols = [condition + '_avg{}'.format(datatype) for condition in expConditions]
    cols.extend([condition + '_{}_log2FC'.format(datatype) for condition in expConditions])
    cols.extend([condition + '_{}_rawTtest_pval'.format(datatype) for condition in expConditions])
    cols.extend([condition + '_{}_fdrBH_corrected_pval'.format(datatype) for condition in expConditions])
    display(df_master[cols].head(n=10))

    # save top 5 most significant genes to genes_of_interest --> gene window plot
    for gene in df_master.index[:5]:
        if gene not in significant_genes:
           significant_genes.append(gene)
        


#### 11. Plot correlation between replicates (within condition)
> if there are > 2 samples in a condition, only plots correlation between the first 2 listed

In [ ]:
plot.gene_intraConditionCorrelation(df_master, user_inputs, plot_dir, RNAseq)

#### 12. Plot correlation between conditions (sample vs WT)
> Without filtering for quality/expression/coverage/etc, expect relatively high correlation between conditions 
>
>** only plots sample vs WT, not sample vs sample

In [ ]:
plot.gene_interConditionCorrelation(df_master, user_inputs, plot_dir, RNAseq)

#### 13. Plot histogram of log2 fold change for sample vs WT
> Without filtering for quality/expression/coverage/etc, expect more of a bell curve around 0


In [ ]:
plot.gene_histFC(df_master, user_inputs, plot_dir, RNAseq)

#### 14. Filter to exclude low expression & coverage genes

> **plots below this point will be more useful**
>
> using the same filter method as during metagene analysis, with minCDS defined in user_inputs

In [ ]:
#filter

# save an unfiltered copy of feature_dict_meta just in case
feature_dict_meta_full = copy.deepcopy(feature_dict_meta)

# remove genes which have low expression/coverage (>80% 0 or < 0.5 read average across CDS of gene) or have < 100nt CDS 
feature_dict_meta = analysis.filter_genes(sample_names, feature_dict_meta, user_inputs)


#### 15. Create a reference of common genes

>  genes common to all samples within a condition <br>
>  genes common to all samples within a condition AND WT together

In [ ]:
genes_in_all_samples, genes_per_condition, genes_per_conditionPlusWT = analysis.common_genes(sample_names, feature_dict_meta, genome_dict, user_inputs, RNAseq)


#### 16. Filter by common genes & replot
> for each condition, filter by common genes for all condition + WT samples <br>
> Replot inter & intra condition correlation and histograms <br>
> save to a subfolder specifying which condition was used to filter

In [ ]:
WT = ['DMSO' if 'DMSO' in user_inputs['condition_info'] else 'WT'][0]
expConditions = [condition  for condition in user_inputs['condition_info'] if (condition not in ['WT', 'DMSO'] and user_inputs['condition_type'][condition]=='RiboSeq')]


# filter datatable & replot
for condition in expConditions:
    samples_to_consider=  user_inputs['condition_info'][WT] + user_inputs['condition_info'][condition]
    genes_to_consider = genes_per_conditionPlusWT[condition]
    user_inputs_filt = copy.deepcopy(user_inputs)
    user_inputs_filt['condition_info'] = { WT: user_inputs['condition_info'][WT], condition: user_inputs['condition_info'][condition] }

    filt_dir = profiling_dir + 'filt_%s_plots/' % condition
    if not os.path.exists(filt_dir):
        os.mkdir(filt_dir)

    df_master_filt = pd.DataFrame()

    for col in df_master:
        for expSample in user_inputs['condition_info'][condition] + user_inputs['condition_info'][WT]:
            if col in [expSample + '_RPKM', expSample + '_TE']:
                df_master_filt[col] = df_master[col][genes_to_consider]
            if expSample in user_inputs['rnaSeq_pairs'].keys():
                if col in [user_inputs['rnaSeq_pairs'][expSample]+'_RPKM']:
                    df_master_filt[col] = df_master[col][genes_to_consider]
        if col.startswith(condition+'_') or col.startswith(WT+'_'):
            df_master_filt[col] = df_master[col][genes_to_consider]

    df_master_filt.sort_values(condition + '_%s_rawTtest_pval' % datatype, inplace=True)
    display(df_master_filt)
    plot.gene_intraConditionCorrelation(df_master_filt, user_inputs_filt, filt_dir, RNAseq)
    plot.gene_interConditionCorrelation(df_master_filt, user_inputs_filt, filt_dir, RNAseq)
    plot.gene_histFC(df_master_filt, user_inputs_filt, filt_dir, RNAseq)
    # save the dataframe to csv
    df_master_filt.to_csv('{}/gene_profiling_master_table_filt-by-{}_{}_offset_{}.csv'.format(filt_dir, condition, user_inputs['alignment_type'], user_inputs['mapping_offset']))



#### 17. Plot gene window for top significant genes
> plots whole-gene window for top 5 significant genes in each sample, pulled during step 10 <br>
> if a gene is top 5 for more than one sample, it will only be plotted once

In [ ]:
reload(plot)
for gene in significant_genes:

    #supply full gene name to plot reads for that gene
    #plot.whole_gene_window_rpkm(gene, sample_names, total_read_dict, feature_dict_meta_full, genome_dict, gene_dir, user_inputs)
    
    if RNAseq == True:
        #plot.whole_gene_window_TE(gene, sample_names, total_read_dict, feature_dict_meta_full, genome_dict, gene_dir, user_inputs, RNAseq)
        plot.stack_plot_TE(gene,[condition for condition in user_inputs['condition_info'] if user_inputs['condition_type'][condition] == 'RiboSeq'],
                           plot_dir, feature_dict_meta_full, total_read_dict, genome_dict, user_inputs)
    else: 
        
        plot.stack_plot( gene,[condition for condition in user_inputs['condition_info']],
                           plot_dir, feature_dict_meta_full, total_read_dict, genome_dict, user_inputs)
        

#### 18. Close out & save analysis

> **From here, move to RiboSeq_CodonAnalysis.ipynb**

In [ ]:
endtime = time.time()
print('Gene profiling analysis complete on {} samples in {} hh:mm:ss - alignment type: {}, mapping_offset: {}'.format(len(sample_names), \
            time.strftime("%H:%M:%S", time.gmtime(endtime-starttime)), user_inputs['alignment_type'], user_inputs['mapping_offset']))


In [ ]:
# sleep to allow notebook time to autosave all outputs 
time.sleep(120)
# produce an html copy of the notebook ; this is your log of the run & contains all code + cell outputs + plots 
os.system("jupyter nbconvert --to html RiboSeq_GeneProfilingAnalysis.ipynb")
os.rename('RiboSeq_GeneProfilingAnalysis.html', 
          profiling_dir + 'RiboSeq_GeneProfilingAnalysis_notebookLog_{}Alignment_{}offset.html'.format(user_inputs['alignment_type'], user_inputs['mapping_offset']))



# **Next: RiboSeq_CodonAnalysis.ipynb**